# SEC EDGAR 재무제표 수집 (IS 항목 분기별 변환 개선)

In [1]:
import sys
import pandas as pd
import numpy as np
import time

# 모듈 임포트
sys.path.append(r"/US_Market/collect")

from sec_data_pipeline.collectors.get_us_ticker import get_filtered_us_tickers
from sec_data_pipeline.collectors.rate_limiter import AdaptiveRateLimiter
from sec_data_pipeline.valuation.integrated_financial_analyzer_mysql_fixed import IntegratedFinancialAnalyzer
from sec_data_pipeline.storage.db_manager import DBManager
from DATA.stock_invest_function import get_db_host

# 설정
START_DATE = "2025-01-01"
MAX_TICKERS = 5000
OFFSET = 0

# DB 초기화
db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}
db_manager = DBManager(db_info)
analyzer = IntegratedFinancialAnalyzer(db_info=db_info, wrds_conn_str="여기에_WRDS_주소")

rate_limiter = AdaptiveRateLimiter(
    max_calls=8,
    time_window=1.0,
    min_calls=3,
    backoff_factor=0.8,
)

headers = {"User-Agent": "Hoyoung Research <stox1224@gmail.com>"}

ALL_TICKERS = get_filtered_us_tickers()
TICKER_LIST = ALL_TICKERS[OFFSET : OFFSET + MAX_TICKERS]
TICKER_LIST = ['GOOG']

print(f"총 {len(TICKER_LIST)}개의 티커를 수집합니다. (기준일: {START_DATE})")

✓ DB connected: 192.168.0.230:3307/investar


100%|██████████| 304/304 [00:00<00:00, 579.87it/s]

제외된 기업 수: 1808
남은 기업 수: 5046
티커 수: 5046
총 1개의 티커를 수집합니다. (기준일: 2025-01-01)


## IS 항목 분기별 변환 함수 (개선 버전)

In [2]:
def convert_cumulative_to_quarterly(df):
    """
    IS(손익계산서) 항목의 누적값을 분기별 실제 값으로 변환
    
    SEC EDGAR의 IS 항목들은 다음과 같이 저장됨:
    - Q1: 1분기 실제값
    - Q2: 1분기 + 2분기 누적값
    - Q3: 1분기 + 2분기 + 3분기 누적값
    - Q4: 연간 누적값
    
    이를 각 분기별 실제 값으로 변환:
    - Q1: 그대로 사용
    - Q2: Q2 누적 - Q1 누적
    - Q3: Q3 누적 - Q2 누적
    - Q4: Q4 누적 - Q3 누적
    """
    if df.empty:
        return df
    
    # 데이터 복사 및 정렬
    df_work = df.copy()
    
    # 인덱스를 datetime으로 변환
    if not isinstance(df_work.index, pd.DatetimeIndex):
        df_work.index = pd.to_datetime(df_work.index)
    
    df_work = df_work.sort_index()
    
    # BS(대차대조표) 항목 - 시점 데이터, 변환 불필요
    bs_items = [
        'total_assets', 'current_assets', 'total_liabilities',
        'current_liabilities', 'stockholders_equity', 'cash',
        'long_term_debt', 'inventory', 'accounts_receivable',
        'accounts_payable', 'short_term_debt', 'retained_earnings',
        'property_plant_equipment', 'intangible_assets', 'goodwill'
    ]
    
    # IS(손익계산서) 항목 - 기간 데이터, 누적 → 분기별 변환 필요
    is_items = [
        'revenue', 'net_income', 'operating_income', 'gross_profit',
        'ebit', 'ebitda', 'operating_expense', 'cost_of_revenue',
        'rd_expense', 'sg&a_expense', 'interest_expense',
        'income_tax_expense', 'depreciation_amortization',
        'selling_general_administrative', 'research_development'
    ]
    
    # CF(현금흐름표) 항목 - 기간 데이터, 누적 → 분기별 변환 필요
    cf_items = [
        'operating_cash_flow', 'investing_cash_flow', 'financing_cash_flow',
        'free_cash_flow', 'capex', 'capital_expenditure'
    ]
    
    # 변환 대상 항목 (IS + CF)
    period_items = is_items + cf_items
    
    # 실제 존재하는 변환 대상 컬럼만 필터링
    period_items_in_df = [col for col in period_items if col in df_work.columns]
    
    if not period_items_in_df:
        print("변환 대상 IS/CF 항목이 없습니다.")
        return df_work
    
    print(f"변환 대상 항목: {period_items_in_df}")
    
    # 연도와 분기 정보 추출
    df_work['year'] = df_work.index.year
    df_work['quarter'] = df_work.index.quarter
    
    # 각 연도별로 처리
    for year in sorted(df_work['year'].unique()):
        year_mask = df_work['year'] == year
        year_data = df_work[year_mask].copy()
        
        # 해당 연도의 분기들을 정렬
        quarters = sorted(year_data['quarter'].unique())
        
        print(f"\n{year}년 처리 중... (분기: {quarters})")
        
        for i, quarter in enumerate(quarters):
            # 현재 분기 데이터
            current_mask = (df_work['year'] == year) & (df_work['quarter'] == quarter)
            current_idx = df_work[current_mask].index
            
            if len(current_idx) == 0:
                continue
            
            current_idx = current_idx[0]
            
            if quarter == 1:
                # 1분기는 그대로 사용
                print(f"  Q{quarter}: 누적값 그대로 사용 (1분기)")
                continue
            
            # 이전 분기 찾기
            prev_quarter = quarter - 1
            prev_mask = (df_work['year'] == year) & (df_work['quarter'] == prev_quarter)
            prev_idx_list = df_work[prev_mask].index
            
            if len(prev_idx_list) == 0:
                print(f"  Q{quarter}: 이전 분기(Q{prev_quarter}) 데이터 없음 - 누적값 그대로 사용")
                continue
            
            prev_idx = prev_idx_list[0]
            
            # IS/CF 항목들에 대해 차분 계산
            for item in period_items_in_df:
                current_value = df_work.loc[current_idx, item]
                prev_value = df_work.loc[prev_idx, item]
                
                # 둘 다 유효한 값인 경우에만 차분 계산
                if pd.notna(current_value) and pd.notna(prev_value):
                    quarterly_value = current_value - prev_value
                    df_work.loc[current_idx, item] = quarterly_value
                    
                    # 디버깅 출력 (revenue만)
                    if item == 'revenue':
                        print(f"  Q{quarter} {item}: {current_value:.2e} - {prev_value:.2e} = {quarterly_value:.2e}")
    
    # year, quarter 컬럼 제거
    df_work = df_work.drop(columns=['year', 'quarter'], errors='ignore')
    
    return df_work


print("IS 항목 분기별 변환 함수 정의 완료")

IS 항목 분기별 변환 함수 정의 완료


## 메인 수집 루프

In [3]:
# 메인 루프
for i, ticker in enumerate(TICKER_LIST, start=1):
    print(f"\n{'='*80}")
    print(f"[{i}/{len(TICKER_LIST)}] {ticker} 처리 시작")
    print(f"{'='*80}")

    rate_limiter.wait_if_needed()

    try:
        result = analyzer.analyze(ticker, headers=headers, table_name="us_fundq")
        if result is None:
            print(f"✗ {ticker}: 데이터 없음")
            continue

        final_df, cik, entity_name = result
        
        print(f"\n원본 데이터 shape: {final_df.shape}")
        print(f"원본 데이터 인덱스: {final_df.index.min()} ~ {final_df.index.max()}")

        # 날짜 필터링
        if not final_df.empty:
            final_df.index = pd.to_datetime(final_df.index)
            
            print(f"\n변환 전 Revenue 샘플:")
            if 'revenue' in final_df.columns:
                print(final_df[['revenue']].tail(8))
            
            # IS 항목 누적값 → 분기별 실제 값으로 변환
            print(f"\nIS 항목 분기별 변환 시작...")
            final_df = convert_cumulative_to_quarterly(final_df)
            print(f"IS 항목 분기별 변환 완료")
            
            print(f"\n변환 후 Revenue 샘플:")
            if 'revenue' in final_df.columns:
                print(final_df[['revenue']].tail(8))
            
            # 날짜 필터링 (START_DATE 이후)
            final_df = final_df[final_df.index >= pd.to_datetime(START_DATE)]

        if final_df.empty:
            print(f"⚠ {ticker}: {START_DATE} 이후의 데이터 없음")
            continue

        # DB 저장용 인덱스 정리
        if final_df.index.name != "date":
            final_df.index.name = "date"

        print(f"\n최종 저장 데이터 shape: {final_df.shape}")
        
        db_manager.save_normalized_data(
            ticker=ticker,
            cik=cik,
            df=final_df,
            item_mapping=None
        )
        print(f"\n✓ {ticker} ({entity_name}) {len(final_df)}건 저장 완료")

    except Exception as e:
        import traceback
        msg = str(e)
        if "429" in msg:
            print(f"⚠ {ticker}: SEC 429 감지 → 60초 대기")
            rate_limiter.on_rate_limit_error()
            time.sleep(60)
            continue
        print(f"✗ {ticker} 처리 중 오류:")
        print(traceback.format_exc())
        continue

print("\n" + "="*80)
print("작업 완료")
print("="*80)


[1/1] GOOG 처리 시작
[GOOG] 통합 재무분석 시작
✓ Entity: Alphabet Inc. (CIK: 1652044)
✓ EDGAR 정규화 DF: 48 rows × 31 cols
✓ MySQL 연결 성공: 192.168.0.230:3307/investar
⚠ WRDS 쿼리 실패: Execution failed on sql 'SELECT * FROM us_fundq WHERE tic=%s': (1146, "Table 'investar.us_fundq' doesn't exist")
WRDS 데이터 없음 → EDGAR 그대로 반환
✓ EDGAR+WRDS 병합 DF: 48 rows × 31 cols
재무비율 계산 중...
  - 수익성 비율 계산...
  - 레버리지 비율 계산...
  - 유동성 비율 계산...
  - 효율성 비율 계산...
재무비율 계산 완료!
✓ 최종 DF (비율 포함): 48 rows × 47 cols
[GOOG] 통합 재무분석 종료

원본 데이터 shape: (48, 47)
원본 데이터 인덱스: 2012-12-31 00:00:00 ~ 2025-12-31 00:00:00

변환 전 Revenue 샘플:
                 revenue
date                    
2024-03-31  8.053900e+10
2024-06-30  8.474200e+10
2024-09-30  8.826800e+10
2024-12-31  8.826800e+10
2025-03-31  9.023400e+10
2025-06-30  9.023400e+10
2025-09-30  9.023400e+10
2025-12-31  9.023400e+10

IS 항목 분기별 변환 시작...
변환 대상 항목: ['revenue', 'net_income', 'operating_income', 'cost_of_revenue', 'interest_expense', 'income_tax_expense', 'depreciation_amortization

## 결과 확인

In [4]:
# 저장된 데이터 확인
import pymysql

query = """
SELECT date, revenue, net_income, operating_income, total_assets, current_assets
FROM us_fundq
WHERE ticker = 'GOOG'
  AND date >= '2025-01-01'
ORDER BY date
"""

conn = pymysql.connect(**db_info)
df_check = pd.read_sql(query, conn)
conn.close()

print("\n저장된 데이터:")
print(df_check)

print("\n\nIS 항목 확인 (분기별로 값이 달라야 정상):")
print(df_check[['date', 'revenue', 'net_income', 'operating_income']])

print("\n\nBS 항목 확인 (시점 데이터이므로 분기별로 다름):")
print(df_check[['date', 'total_assets', 'current_assets']])

DatabaseError: Execution failed on sql '
SELECT date, revenue, net_income, operating_income, total_assets, current_assets
FROM us_fundq
WHERE ticker = 'GOOG'
  AND date >= '2025-01-01'
ORDER BY date
': (1146, "Table 'investar.us_fundq' doesn't exist")

## 변환 로직 설명

In [ ]:
print("""
=== IS 항목 누적값 → 분기별 변환 로직 ===

SEC EDGAR의 IS 항목은 연초부터의 누적값으로 저장됩니다:

[변환 전 - 누적값]
2025-Q1: 90.234B  (1분기 매출)
2025-Q2: 90.234B  (1분기 + 2분기 누적 - 하지만 같은 값으로 표시됨)
2025-Q3: 90.234B  (1분기 + 2분기 + 3분기 누적 - 하지만 같은 값으로 표시됨)
2025-Q4: 90.234B  (연간 누적 - 하지만 같은 값으로 표시됨)

[변환 후 - 분기별 실제값]
2025-Q1: 90.234B  (1분기 매출)
2025-Q2: XX.XXXB  (2분기 매출 = Q2누적 - Q1누적)
2025-Q3: YY.YYYB  (3분기 매출 = Q3누적 - Q2누적)
2025-Q4: ZZ.ZZZB  (4분기 매출 = Q4누적 - Q3누적)

변환 대상:
- IS 항목: revenue, net_income, operating_income, gross_profit 등
- CF 항목: operating_cash_flow, investing_cash_flow 등

변환 제외:
- BS 항목: total_assets, current_assets 등 (시점 데이터)
""")